# Notebook 2b — Threshold Analysis (Tables Only)

**Purpose:** Paper-ready **tabular** version of `threshold_analysis.ipynb` — same threshold-sweep information, no charts. All F-measure values are on a 0–100 scale and rounded to two decimals; threshold columns are printed at their raw resolution (2 decimal places).

**Input:** Same `stdout.log` files as `threshold_analysis.ipynb`.

**Definitions:**
- **Detection threshold `k`** — model output cut-off in `evaluate_prototypes` (0.00, 0.05, …, 0.95).
- **Post-proc A (adaptive)** — adaptive threshold sweep at 0.2, 0.3, 0.4, 0.5.
- **Post-proc B (length)** — minimum event-length threshold sweep at 0.05, 0.10, 0.15, 0.20 s.
- All reported F-measures are the official DCASE overall F-measure: the **harmonic mean** of the six per-subset F-measures, each computed from TP/FP/FN pooled within its subset.

**Outputs (all tabular):**
1. F-measure vs detection threshold `k`, no post-processing (k × experiment)
2. F-measure vs post-proc A threshold — best over `k` (thresh_a × experiment)
3. F-measure vs post-proc B threshold — best over `k` (thresh_b × experiment)
4. Full `k` × post-proc threshold matrices per experiment (A and B) with the argmax highlighted
5. Winning threshold configuration per experiment
6. LaTeX export of the winning-threshold table

## 0 — Configuration

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
import parse_logs as pl

# ── Experiment log paths ─────────────────────────────────────────
EXPERIMENT_LOGS = {
    "Baseline":            "/home/helios/Downloads/run_logs/baseline.log",  # e.g. "../logs/experiments/runs/baseline/2026-.../stdout.log"
    "CIFE":                "/home/helios/Downloads/run_logs/CIFE.log",
    "Phase 1 (Random)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/ablation_experiments_aggressive/2026-07-08_20-16-22/stdout.log",
    "Phase 2 (Energy)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/ablation_experiments_aggressive/2026-07-09_02-57-24/stdout.log",
    "Phase 3 (Activation)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/ablation_experiments_aggressive/2026-07-09_09-07-44/stdout.log",
    "AFRI (Direct)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/paper/run_logs/afri_direct_v2.log",
    "AFRI (Inverse)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/paper/run_logs/afri_inverse_v2.log",
}

# Rounding used throughout the notebook (typical for a research paper).
N_DECIMALS = 2

logs_to_parse = {k: v for k, v in EXPERIMENT_LOGS.items() if v is not None}
print(f"Parsing {len(logs_to_parse)} experiment(s)...")

# ── Paper tables output directory ──────────────────────────
# Every table produced below is also written as a stand-alone .tex file
# (\begin{table} block) under paper/tables/.
PAPER_TABLES_DIR = Path("../../../paper/tables").resolve()
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

def save_tex(df, filename, caption, label, **to_latex_kwargs):
    """Write `df` as a standalone LaTeX table to PAPER_TABLES_DIR/filename."""
    kwargs = dict(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=caption,
        label=label,
    )
    kwargs.update(to_latex_kwargs)
    out = PAPER_TABLES_DIR / filename
    out.write_text(df.to_latex(**kwargs))
    print(f"  ✓ Saved {out.name}")


Parsing 7 experiment(s)...


## 1 — Load & Parse Logs

In [2]:
import pandas as pd
import numpy as np

experiments = pl.load_experiments(logs_to_parse)

exp_labels = [k for k in EXPERIMENT_LOGS.keys() if experiments.get(k) is not None]

df_nopost = pl.no_postproc_df(experiments)
df_sweepA = pl.sweep_A_df(experiments)
df_sweepB = pl.sweep_B_df(experiments)
df_final  = pl.final_results_df(experiments)

print(f"No-postproc rows : {len(df_nopost)}")
print(f"Sweep A rows     : {len(df_sweepA)}")
print(f"Sweep B rows     : {len(df_sweepB)}")

  ✓ Baseline                        fmeasure=16.741
  ✓ CIFE                            fmeasure=15.217
  ✓ Phase 1 (Random)                fmeasure=15.489
  ✓ Phase 2 (Energy)                fmeasure=15.443
  ✓ Phase 3 (Activation)            fmeasure=15.447
  ✓ AFRI (Direct)                   fmeasure=12.073
  ✓ AFRI (Inverse)                  fmeasure=15.587
No-postproc rows : 133
Sweep A rows     : 532
Sweep B rows     : 560


## 2 — F-measure vs Detection Threshold `k` (No Post-processing)

Tabular replacement for the line plot in Section 2 of the original notebook. Rows are detection thresholds `k`, columns are experiments. Trailing `Best` row = per-experiment argmax over `k`.

In [3]:
if df_nopost.empty:
    print("No-postproc data not available.")
else:
    pivot_k = (
        df_nopost.pivot_table(index="k", columns="experiment", values="fmeasure", aggfunc="max")
        .reindex(columns=exp_labels)
        .sort_index()
        .round(N_DECIMALS)
    )
    # Append summary rows: max F over k, and the argmax k
    best_f = pivot_k.max().round(N_DECIMALS)
    best_k = pivot_k.idxmax()
    summary = pd.DataFrame([best_f, best_k], index=["Best F", "Best k"])
    pivot_k_out = pd.concat([pivot_k, summary])

    print("=== Table 1 — F-measure vs Detection Threshold k (No Post-proc, 0–100) ===")
    display(pivot_k_out)

    save_tex(
        pivot_k_out,
        "threshold_k_sweep_no_postproc.tex",
        caption=(
            "F-measure (harmonic subset mean) as a function of the detection threshold $k$ "
            "without any post-processing. The last two rows report the "
            "per-experiment best F and the argmax $k$. "
            "All F-measure values on a 0--100 scale."
        ),
        label="tab:threshold_k_sweep_no_postproc",
    )

=== Table 1 — F-measure vs Detection Threshold k (No Post-proc, 0–100) ===


experiment,Baseline,CIFE,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse)
0.0,7.72,6.99,6.72,6.74,6.64,5.07,6.72
0.05,8.09,7.11,6.93,7.06,6.87,5.36,6.95
0.1,8.38,7.27,7.31,7.47,7.24,5.62,7.48
0.15,8.61,7.45,7.44,7.57,7.58,5.75,7.54
0.2,8.69,7.35,7.89,7.54,7.54,5.94,7.94
0.25,8.84,7.68,7.91,8.12,7.87,5.99,7.78
0.3,9.24,8.42,8.59,8.62,8.26,6.30,8.61
0.35,9.66,8.11,8.40,8.73,8.37,6.22,8.43
0.4,9.84,8.42,8.58,8.50,8.49,6.41,8.60
0.45,10.06,8.46,8.55,8.35,8.48,6.55,8.61


  ✓ Saved threshold_k_sweep_no_postproc.tex


## 3 — F-measure vs Adaptive Post-proc Threshold A

For each (experiment, thresh_a) pair, we report the **best F over all detection thresholds `k`** — the same aggregation as the line plot in Section 3 of the original notebook.

In [4]:
if df_sweepA.empty:
    print("Sweep A data not available.")
else:
    df_A_agg = (
        df_sweepA.groupby(["experiment", "thresh_a"])["fmeasure"]
        .max()
        .reset_index()
    )
    pivot_A = (
        df_A_agg.pivot_table(index="thresh_a", columns="experiment", values="fmeasure")
        .reindex(columns=exp_labels)
        .sort_index()
        .round(N_DECIMALS)
    )
    best_f = pivot_A.max().round(N_DECIMALS)
    best_a = pivot_A.idxmax()
    summary = pd.DataFrame([best_f, best_a], index=["Best F", "Best thresh_a"])
    pivot_A_out = pd.concat([pivot_A, summary])

    print("=== Table 2 — F-measure vs Adaptive Post-proc Threshold A (best over k, 0–100) ===")
    display(pivot_A_out)

    save_tex(
        pivot_A_out,
        "threshold_sweep_A.tex",
        caption=(
            "F-measure (harmonic subset mean) as a function of the adaptive post-processing "
            "threshold (type A). For each cell we take the best F across "
            "detection thresholds $k$. The last two rows report the "
            "per-experiment best F and the argmax threshold. "
            "All F-measure values on a 0--100 scale."
        ),
        label="tab:threshold_sweep_A",
    )

=== Table 2 — F-measure vs Adaptive Post-proc Threshold A (best over k, 0–100) ===


experiment,Baseline,CIFE,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse)
0.2,11.47,9.11,10.40,10.47,10.42,8.31,10.56
0.30000000000000004,11.83,9.35,10.70,10.76,10.70,8.48,10.86
0.4000000000000001,12.03,9.47,10.85,10.91,10.86,8.58,11.01
0.5000000000000001,12.33,9.64,11.08,11.14,11.08,8.71,11.25
Best F,12.33,9.64,11.08,11.14,11.08,8.71,11.25
Best thresh_a,0.50,0.50,0.50,0.50,0.50,0.50,0.50


  ✓ Saved threshold_sweep_A.tex


## 4 — F-measure vs Length Post-proc Threshold B

Same aggregation as Section 3, but for the length-based post-processor (`thresh_b`, seconds).

In [5]:
if df_sweepB.empty:
    print("Sweep B data not available.")
else:
    df_B_agg = (
        df_sweepB.groupby(["experiment", "thresh_b"])["fmeasure"]
        .max()
        .reset_index()
    )
    pivot_B = (
        df_B_agg.pivot_table(index="thresh_b", columns="experiment", values="fmeasure")
        .reindex(columns=exp_labels)
        .sort_index()
        .round(N_DECIMALS)
    )
    best_f = pivot_B.max().round(N_DECIMALS)
    best_b = pivot_B.idxmax()
    summary = pd.DataFrame([best_f, best_b], index=["Best F", "Best thresh_b"])
    pivot_B_out = pd.concat([pivot_B, summary])

    print("=== Table 3 — F-measure vs Length Post-proc Threshold B (best over k, 0–100) ===")
    display(pivot_B_out)

    save_tex(
        pivot_B_out,
        "threshold_sweep_B.tex",
        caption=(
            "F-measure (harmonic subset mean) as a function of the minimum event-length "
            "post-processing threshold (type B, in seconds). For each cell "
            "we take the best F across detection thresholds $k$. The last two "
            "rows report the per-experiment best F and the argmax threshold. "
            "All F-measure values on a 0--100 scale."
        ),
        label="tab:threshold_sweep_B",
    )

=== Table 3 — F-measure vs Length Post-proc Threshold B (best over k, 0–100) ===


experiment,Baseline,CIFE,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse)
0.05,15.30,13.62,13.68,13.77,13.56,10.33,13.70
0.1,16.47,15.12,15.44,15.44,15.40,12.01,15.59
0.15000000000000002,16.74,15.22,15.49,15.28,15.45,12.07,15.56
0.2,14.73,13.73,12.89,12.79,12.75,9.52,12.93
Best F,16.74,15.22,15.49,15.44,15.45,12.07,15.59
Best thresh_b,0.15,0.15,0.15,0.10,0.15,0.15,0.10


  ✓ Saved threshold_sweep_B.tex


## 5 — Full `k` × Post-proc Threshold Matrices

Tabular replacement for the per-experiment heatmaps. For each experiment we print two tables:
- **Post-proc A** — rows are detection threshold `k`, columns are `thresh_a`.
- **Post-proc B** — rows are detection threshold `k`, columns are `thresh_b`.

The argmax cell in each table is reported in the printed caption.

In [6]:
def _matrix(df, k_col, pp_col):
    if df.empty:
        return None
    m = df.pivot_table(index=k_col, columns=pp_col, values="fmeasure", aggfunc="max")
    return m.sort_index().sort_index(axis=1).round(N_DECIMALS)

def _argmax_cell(m):
    if m is None or m.empty or m.values.size == 0:
        return None, None, None
    flat = np.nanargmax(m.values)
    r, c = divmod(flat, m.values.shape[1])
    return m.index[r], m.columns[c], m.values[r, c]

def _slug(text):
    return (
        text.lower()
        .replace(" ", "_").replace("+", "plus").replace("(", "").replace(")", "")
        .replace("/", "_").replace("-", "_")
    )

for label in exp_labels:
    print("=" * 78)
    print(f"Experiment: {label}")
    print("=" * 78)

    sub_A = df_sweepA[df_sweepA["experiment"] == label] if not df_sweepA.empty else pd.DataFrame()
    sub_B = df_sweepB[df_sweepB["experiment"] == label] if not df_sweepB.empty else pd.DataFrame()

    mA = _matrix(sub_A, "k", "thresh_a")
    if mA is not None and not mA.empty:
        kA, aA, fA = _argmax_cell(mA)
        print(f"  Post-proc A (adaptive) — argmax: F = {fA:.{N_DECIMALS}f} at k = {kA}, thresh_a = {aA}")
        display(mA)
        save_tex(
            mA,
            f"threshold_matrix_A_{_slug(label)}.tex",
            caption=(
                f"F-measure (harmonic subset mean) heatmap for the {label} configuration "
                f"across detection threshold $k$ (rows) and adaptive "
                f"post-processing threshold (type A, columns). "
                f"Argmax: $F = {fA:.{N_DECIMALS}f}$ at $k = {kA}$, "
                f"$\\mathrm{{thresh}}_A = {aA}$. All F values on 0--100 scale."
            ),
            label=f"tab:threshold_matrix_A_{_slug(label)}",
        )
    else:
        print("  Post-proc A: no data.")

    mB = _matrix(sub_B, "k", "thresh_b")
    if mB is not None and not mB.empty:
        kB, bB, fB = _argmax_cell(mB)
        print(f"  Post-proc B (length)   — argmax: F = {fB:.{N_DECIMALS}f} at k = {kB}, thresh_b = {bB}")
        display(mB)
        save_tex(
            mB,
            f"threshold_matrix_B_{_slug(label)}.tex",
            caption=(
                f"F-measure (harmonic subset mean) heatmap for the {label} configuration "
                f"across detection threshold $k$ (rows) and minimum "
                f"event-length post-processing threshold (type B, seconds, "
                f"columns). Argmax: $F = {fB:.{N_DECIMALS}f}$ at $k = {kB}$, "
                f"$\\mathrm{{thresh}}_B = {bB}\\,$s. "
                f"All F values on 0--100 scale."
            ),
            label=f"tab:threshold_matrix_B_{_slug(label)}",
        )
    else:
        print("  Post-proc B: no data.")

Experiment: Baseline
  Post-proc A (adaptive) — argmax: F = 12.33 at k = 0.9, thresh_a = 0.5000000000000001


thresh_a,0.2,0.3,0.4,0.5
k,,,,
0.00,7.80,7.98,8.07,8.19
0.05,8.18,8.37,8.46,8.60
0.10,8.48,8.68,8.78,8.93
0.15,8.71,8.93,9.03,9.19
0.20,8.79,9.01,9.12,9.28
0.25,8.94,9.17,9.28,9.45
0.30,9.35,9.58,9.71,9.89
0.35,9.78,10.04,10.18,10.37
0.40,9.96,10.23,10.37,10.57


  ✓ Saved threshold_matrix_A_baseline.tex
  Post-proc B (length)   — argmax: F = 16.74 at k = 0.8, thresh_b = 0.15000000000000002


thresh_b,0.05,0.10,0.15,0.20
k,,,,
0.00,10.62,13.00,13.54,12.72
0.05,11.14,13.48,14.07,12.82
0.10,11.59,14.02,14.53,13.08
0.15,11.91,14.31,14.71,13.18
0.20,12.06,14.50,14.75,13.02
0.25,12.42,14.78,14.86,13.06
0.30,13.04,15.35,15.33,13.16
0.35,13.48,15.70,15.62,13.41
0.40,13.65,15.72,15.82,13.80


  ✓ Saved threshold_matrix_B_baseline.tex
Experiment: CIFE
  Post-proc A (adaptive) — argmax: F = 9.64 at k = 0.9, thresh_a = 0.5000000000000001


thresh_a,0.2,0.3,0.4,0.5
k,,,,
0.00,7.05,7.21,7.28,7.38
0.05,7.17,7.33,7.40,7.51
0.10,7.34,7.51,7.58,7.69
0.15,7.52,7.69,7.77,7.89
0.20,7.42,7.58,7.66,7.78
0.25,7.76,7.94,8.02,8.15
0.30,8.51,8.73,8.84,8.99
0.35,8.19,8.40,8.50,8.64
0.40,8.51,8.73,8.84,8.99


... (19 outputs left out)

thresh_a,0.2,0.3,0.4,0.5
k,,,,
0.00,6.78,6.92,6.99,7.09
0.05,7.01,7.16,7.23,7.33
0.10,7.55,7.71,7.79,7.91
0.15,7.62,7.78,7.86,7.99
0.20,8.02,8.21,8.30,8.44
0.25,7.86,8.03,8.12,8.26
0.30,8.70,8.92,9.02,9.18
0.35,8.52,8.72,8.82,8.98
0.40,8.69,8.90,9.00,9.16


  ✓ Saved threshold_matrix_A_afri_inverse.tex
  Post-proc B (length)   — argmax: F = 15.59 at k = 0.9, thresh_b = 0.1


thresh_b,0.05,0.10,0.15,0.20
k,,,,
0.00,8.97,11.62,11.99,11.44
0.05,9.34,12.13,12.26,11.16
0.10,10.02,12.83,12.63,11.40
0.15,10.15,12.94,12.59,11.17
0.20,10.58,13.40,13.34,11.78
0.25,10.55,13.36,13.29,11.23
0.30,11.49,14.27,14.06,12.06
0.35,11.33,14.08,13.85,11.96
0.40,11.51,14.00,14.06,12.08


  ✓ Saved threshold_matrix_B_afri_inverse.tex


## 6 — Winning Threshold Configuration per Experiment

For each experiment: the final best F-measure, the winning detection threshold `k`, the winning post-processing threshold, and which post-processor produced it (A = adaptive, B = length).

In [7]:
rows = []
for label in exp_labels:
    data = experiments.get(label)
    if data is None:
        continue
    final_fm = data["final"].get("fmeasure")
    winning_k = None
    winning_thresh = None
    proc_type = None
    for entry in data["best_A"]:
        if entry.get("fmeasure") == final_fm:
            winning_k = entry["k"]
            winning_thresh = entry["winning_thresh"]
            proc_type = "A (adaptive)"
            break
    if proc_type is None:
        for entry in data["best_B"]:
            if entry.get("fmeasure") == final_fm:
                winning_k = entry["k"]
                winning_thresh = entry["winning_thresh"]
                proc_type = "B (length)"
                break
    rows.append({
        "Experiment": label,
        "Best F-measure": round(final_fm, N_DECIMALS) if final_fm is not None else None,
        "Winning k": winning_k,
        "Winning post-proc thresh": winning_thresh,
        "Post-proc type": proc_type,
    })

df_winning = pd.DataFrame(rows).set_index("Experiment")

print("=== Table 5 — Winning Threshold Configuration per Experiment ===")
display(df_winning)

save_tex(
    df_winning,
    "threshold_winning.tex",
    caption=(
        "Winning threshold configuration per experiment. "
        "$k$ is the detection threshold applied to the model output; the "
        "post-processing threshold is either the adaptive threshold "
        "(type A) or the minimum event-length threshold in seconds "
        "(type B). F-measure on a 0--100 scale."
    ),
    label="tab:threshold_winning",
    bold_rows=True,
)

=== Table 5 — Winning Threshold Configuration per Experiment ===


,Best F-measure,Winning k,Winning post-proc thresh,Post-proc type
Experiment,,,,
Baseline,16.74,0.80,0.15,B (length)
CIFE,15.22,0.65,0.15,B (length)
Phase 1 (Random),15.49,0.90,0.15,B (length)
Phase 2 (Energy),15.44,0.85,0.10,B (length)
Phase 3 (Activation),15.45,0.90,0.15,B (length)
AFRI (Direct),12.07,0.90,0.15,B (length)
AFRI (Inverse),15.59,0.90,0.10,B (length)


  ✓ Saved threshold_winning.tex


## 7 — LaTeX Export

Copy-paste ready `\begin{table}` blocks for a research paper: the winning-threshold table (Table 5) and the no-postproc `k` sweep (Table 1).

In [8]:
if not df_winning.empty:
    latex_win = df_winning.to_latex(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=(
            "Winning threshold configuration per experiment. "
            "$k$ is the detection threshold applied to model output; "
            "the post-processing threshold is either the adaptive threshold (A) or "
            "the minimum event-length threshold in seconds (B). "
            "F-measure is on a 0--100 scale."
        ),
        label="tab:winning_thresholds",
        bold_rows=True,
    )
    print("% ==== Winning thresholds ====")
    print(latex_win)

if not df_nopost.empty:
    latex_k = pivot_k_out.to_latex(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=(
            "F-measure (harmonic subset mean) as a function of the detection threshold $k$ "
            "without any post-processing. Last two rows report the per-experiment "
            "best F and the argmax $k$. All F values on a 0--100 scale."
        ),
        label="tab:k_sweep_nopost",
    )
    print("\n% ==== Detection-threshold sweep, no post-proc ====")
    print(latex_k)

% ==== Winning thresholds ====
\begin{table}
\centering
\caption{Winning threshold configuration per experiment. $k$ is the detection threshold applied to model output; the post-processing threshold is either the adaptive threshold (A) or the minimum event-length threshold in seconds (B). F-measure is on a 0--100 scale.}
\label{tab:winning_thresholds}
\begin{tabular}{lrrrl}
\toprule
{} &  Best F-measure &  Winning k &  Winning post-proc thresh & Post-proc type \\
\textbf{Experiment          } &                 &            &                           &                \\
\midrule
\textbf{Baseline            } &           16.74 &       0.80 &                      0.15 &     B (length) \\
\textbf{CIFE                } &           15.22 &       0.65 &                      0.15 &     B (length) \\
\textbf{Phase 1 (Random)    } &           15.49 &       0.90 &                      0.15 &     B (length) \\
\textbf{Phase 2 (Energy)    } &           15.44 &       0.85 &                      0.1